In [10]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from itertools import combinations
from pathlib import Path
import numpy as np
import os

In [11]:
data_pipeline = "nums_to_cats"
input_pipeline = "baseline"

In [12]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [13]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [14]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [15]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
dtypes: category(1), float64(11)
memory usage: 58.7 MB


In [16]:
cat_columns = X.select_dtypes(include=['category']).columns
num_columns = X.select_dtypes(include=['float64']).columns
num_columns

Index(['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
       'work_study_hours', 'sleep_hours', 'notifications_per_day',
       'app_opens_per_day', 'weekend_screen_time', 'stress_level',
       'academic_work_impact'],
      dtype='str')

In [17]:
drop_columns = X.columns
num_columns = X.select_dtypes(include=['float64'])

for frame in [X, X_test]:
    for col in num_columns:
        frame[f"{col}_cat"] = frame[col].astype('category')

    frame.drop(drop_columns, axis=1, inplace=True)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   age_cat                      662440 non-null  category
 1   daily_screen_time_hours_cat  595515 non-null  category
 2   social_media_hours_cat       557374 non-null  category
 3   gaming_hours_cat             564548 non-null  category
 4   work_study_hours_cat         639851 non-null  category
 5   sleep_hours_cat              646889 non-null  category
 6   notifications_per_day_cat    623785 non-null  category
 7   app_opens_per_day_cat        610659 non-null  category
 8   weekend_screen_time_cat      579306 non-null  category
 9   stress_level_cat             636221 non-null  category
 10  academic_work_impact_cat     647145 non-null  category
dtypes: category(11)
memory usage: 12.6 MB


In [18]:
write(experiment_path / f"train.parq", X)
write(experiment_path / f"test.parq", X_test)